# 05 - Freeze Roster v2 and Create Final Incident-Level Partitions

Controlled step only: freeze the reviewed v2 roster and create the v2 incident-level train, validation, and final holdout split. No MRT download, Normal labeling, feature extraction, model training, or window cap is introduced here.

## Final Holdout Protection

The Final Holdout must never be used for feature selection, feature ablation decisions, Normal-rule threshold tuning, class/window balancing decisions, hyperparameter tuning, Stage 1 threshold selection, or model selection. It is used only after the complete pipeline is frozen.

In [ ]:
from pathlib import Path
import csv
import sys
from collections import Counter

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent.resolve()
sys.path.insert(0, str(ROOT / 'src'))

from bgp_incident_detection.partitioning_v2 import RANDOM_SEED, run, overlap_counts

def read_csv(path):
    with path.open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

print(f'Using fixed random seed: {RANDOM_SEED}')

## Freeze-Readiness Evidence

In [ ]:
readiness = read_csv(ROOT / 'data/roster_v2/roster_v2_freeze_readiness.csv')
readiness

## Generate Frozen Roster and v2 Partitions

In [ ]:
results = run()
assignments = results['assignments']
Counter(row['partition'] for row in assignments)

## Roster and Partition Counts

In [ ]:
frozen_roster = read_csv(ROOT / 'data/roster_v2/frozen_incident_roster_v2.csv')
manifest = read_csv(ROOT / 'data/partitions_v2/frozen_partition_manifest_v2.csv')

print('Frozen roster rows:', len(frozen_roster))
print('Frozen roster class counts:', Counter(row['category'] for row in frozen_roster))
print('Manifest rows:', len(manifest))
print('Manifest partition counts:', Counter(row['partition'] for row in manifest))
print('Manifest class-by-partition counts:')
Counter((row['partition'], row['category']) for row in manifest)

## Overlap Proof

In [ ]:
overlap_counts(manifest)

In [ ]:
manifest_uids = [row['incident_uid'] for row in manifest]
roster_uids = {row['incident_uid'] for row in frozen_roster}

print('Every incident appears exactly once:', len(manifest_uids) == len(set(manifest_uids)) == len(roster_uids) == 1419)
print('Manifest matches frozen roster:', set(manifest_uids) == roster_uids)

## Distribution Summaries

In [ ]:
read_csv(ROOT / 'data/partitions_v2/partition_v2_distribution_summary.csv')

## Final Audit

In [ ]:
audit = read_csv(ROOT / 'data/partitions_v2/partition_v2_quality_audit.csv')
Counter(row['status'] for row in audit), audit